In [1]:
# extract submission with keywords
import json
import re
import time
import os
from IPython.display import clear_output

in_path  = r"D:\ERP\code\RS_2019-04\RS_2019-04"
out_path = r"F:\ERP_data\filtered_MSJG_RS.jsonl"

# M: Mathematics
M = re.compile(
    r"\b("
    r"maths|"
    r"mathematics|"
    r"algebra|"
    r"calculus|"
    r"geometry|"
    r"math\s+(class|major|degree|department|education)"
    r")\b",
    re.I
)

# S: STEM
S = re.compile(
    r"\b("
    r"stem|"
    r"engineering|"
    r"physics|"
    r"engineer|"
    r"computer\s*science|"
    r"cs\s*major|"
    r"programming|"
    r"coding|"
    r"coder|"
    r"software\s*engineer|"
    r"software\s*developer|"
    r"data\s*science|"
    r"data\s*scientist|"
    r"machine\s*learning|"
    r"artificial\s*intelligence|"
    r"women\s*in\s*tech|"
    r"girls\s*who\s*code"
    r")\b",
    re.I
)

# G: Gender
G = re.compile(
    r"\b("
    r"gender|"
    r"sexism|"
    r"sexist|"
    r"discrimination|"
    r"women|"
    r"woman|"
    r"girls|"
    r"female|"
    r"underrepresented|"
    r"male\s*dominated|"
    r"gender\s*gap"
    r")\b",
    re.I
)

report_every_lines = 400000
report_every_seconds = 15

def context_filter(text):
    text_lower = text.lower()
    sentences = re.split(
        r'[.!?]+',
        text_lower
    )
    # sentence level
    for sent in sentences:
        if (
            (M.search(sent) or S.search(sent))
            and G.search(sent)
        ):
            return True
    # comment level
    if (
        (M.search(text_lower) or S.search(text_lower))
        and G.search(text_lower)
    ):
        return True
    return False

def format_bytes(num_bytes):
    num_bytes = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if num_bytes < 1024:
            return f"{num_bytes:.2f} {unit}"
        num_bytes /= 1024
    return f"{num_bytes:.2f} PB"

def format_seconds(seconds):
    if seconds == float("inf"):
        return "inf"
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:d}:{m:02d}:{s:02d}"

total_size = os.path.getsize(in_path)
scanned_bytes = 0
scanned_lines = 0
kept = 0
start_time = time.time()
last_report_time = start_time

clear_output(wait=True)
print("Start filtering...\n")
print(f"Scanned: 0.00 GB / {format_bytes(total_size)} (0.00%)")
print("Scanned lines: 0")
print("Kept lines: 0")
print("Speed: 0.00 MB/s")
print("ETA: calculating...")

# Main loop
with open(in_path, "rb") as fin, open(out_path, "wb") as fout:
    for raw_line in fin:
        scanned_lines += 1
        scanned_bytes += len(raw_line)
        try:
            line = raw_line.decode("utf-8", errors="replace")
            obj = json.loads(line)
        except Exception:
            continue
        title = obj.get("title") or ""
        selftext = obj.get("selftext") or ""
        text = title + " " + selftext

        if text in ("[deleted]", "[removed]"):
            pass
        elif "I am a bot" in text or "AUTOMOD" in text:
            pass
        elif context_filter(text):
            fout.write(raw_line)
            kept += 1

        now = time.time()
        need_report = (
            scanned_lines % report_every_lines == 0
            or (now - last_report_time) >= report_every_seconds
        )

        if need_report:
            elapsed = now - start_time
            pct = (scanned_bytes / total_size) * 100 if total_size else 0
            speed = scanned_bytes / elapsed if elapsed > 0 else 0
            eta = (total_size - scanned_bytes) / speed if speed > 0 else float("inf")

            clear_output(wait=True)
            print("Filtering in progress...\n")
            print(
                f"Scanned: {format_bytes(scanned_bytes)} / {format_bytes(total_size)} "
                f"({pct:.2f}%)"
            )
            print(f"Scanned lines: {scanned_lines:,}")
            print(f"Kept lines: {kept:,}")
            print(f"Speed: {format_bytes(speed)}/s")
            print(f"ETA: {format_seconds(eta)}")
            last_report_time = now

elapsed = time.time() - start_time
pct = (scanned_bytes / total_size) * 100 if total_size else 0
out_size = os.path.getsize(out_path) if os.path.exists(out_path) else 0
ratio = (out_size / total_size) * 100 if total_size else 0
avg_speed = scanned_bytes / elapsed if elapsed > 0 else 0

clear_output(wait=True)
print("Done.\n")
print(
    f"Scanned: {format_bytes(scanned_bytes)} / {format_bytes(total_size)} "
    f"({pct:.2f}%)"
)
print(f"Scanned lines: {scanned_lines:,}")
print(f"Kept lines: {kept:,}")
print(f"Output size: {format_bytes(out_size)}")
print(f"Output/Input ratio: {ratio:.4f}%")
print(f"Average speed: {format_bytes(avg_speed)}/s")
print(f"Saved: {out_path}")
print(f"Total time: {format_seconds(elapsed)}")

Done.

Scanned: 54.46 GB / 54.46 GB (100.00%)
Scanned lines: 18,310,157
Kept lines: 3,255
Output size: 23.68 MB
Output/Input ratio: 0.0425%
Average speed: 23.51 MB/s
Saved: F:\ERP_data\filtered_MSJG_RS.jsonl
Total time: 0:39:31


In [4]:
# classifier
import pandas as pd
import json
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

train_path = r"F:\ERP_data\manual_check_sample_7_28.xlsx"
train_df = pd.read_excel(train_path)
train_df.head()

,id,body,subreddit,relevant,reason
0,1,Good for you on being invited to attend the co...,cscareerquestions,1,Gender-STEM narrative
1,2,He just wants you as a fuck buddy or a number ...,dating,0,No relevant STEM gender context
2,3,Coincidentally I’ve been meeting a lot of peop...,INTP,0,No relevant STEM gender context
3,4,Well his job could be automated by artificial ...,politics,0,No relevant STEM gender context
4,5,Full contents: \n\n**Prices**\n* $1 minimum f...,GameDeals,0,No relevant STEM gender context


In [5]:
#TF-IDF文本转换
X_text = train_df["body"].fillna("")

y = train_df["relevant"]

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2,
    max_features=10000
)

X_train = vectorizer.fit_transform(X_text)
print(X_train.shape)

(200, 2813)


In [6]:
# train Logistic Regression
clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42)

clf.fit(
    X_train,
    y)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [7]:
# cross-validation
scores = cross_val_score(
    clf,
    X_train,
    y,
    cv=5,
    scoring="f1"
)
print(scores)
print(
    "Mean F1:",
    scores.mean()
)

[0.8        0.6        0.75675676 0.82051282 0.57142857]
Mean F1: 0.7097396297396296


In [8]:
# read candidate corpus
candidate_path = r"F:\ERP_data\filtered_MSJG_RS.jsonl"
texts = []
objects = []
with open(candidate_path,"r",encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        title = obj.get("title") or ""
        selftext = obj.get("selftext") or ""
        text = title + " " + selftext
        texts.append(text)
        objects.append(obj)
print("Submission candidate size:", len(texts))

Submission candidate size: 3255


In [9]:
#predict candidate
X_submission = vectorizer.transform(texts)

submission_prob = clf.predict_proba(X_submission)[:,1]

import pandas as pd

pd.Series(submission_prob).describe()

count    3255.000000
mean        0.427093
std         0.079048
min         0.250542
25%         0.371254
50%         0.412519
75%         0.460002
max         0.824228
dtype: float64

In [8]:
# top p submission
top_idx = submission_prob.argsort()[-20:][::-1]
for i in top_idx:
    print("\nPROB:", round(submission_prob[i],3))
    print(
        "TITLE:",
        objects[i].get("title","")
    )
    print(
        objects[i].get("selftext","")[:300]
    )
    print("-"*80)


PROB: 0.824
TITLE: We should do more to celebrate the women who ARE in STEM rather than criticising how few of them there supposedly are.
[deleted]
--------------------------------------------------------------------------------

PROB: 0.824
TITLE: [OC] Women in STEM

--------------------------------------------------------------------------------

PROB: 0.824
TITLE: Women in STEM

--------------------------------------------------------------------------------

PROB: 0.824
TITLE: Books about women in stem??
[removed]
--------------------------------------------------------------------------------

PROB: 0.824
TITLE: Inspiring Fifty calls for nominations from women in STEM

--------------------------------------------------------------------------------

PROB: 0.824
TITLE: Inspiring Fifty calls for the nomination of women in STEM

--------------------------------------------------------------------------------

PROB: 0.824
TITLE: Inspiring Fifty calls for the nomination of women in ST

In [9]:
# test 0.5
threshold = 0.5
idx = [
    i for i,p in enumerate(submission_prob)
    if p >= threshold
]
print(len(idx))

487


In [1]:
# 50sample, p=0.5
import random
import pandas as pd


sample_idx = random.sample(
    idx,
    min(50,len(idx))
)


data=[]

for i in sample_idx:

    data.append({
        "title": objects[i].get("title",""),
        "selftext": objects[i].get("selftext",""),
        "probability": submission_prob[i]
    })


pd.DataFrame(data).to_excel(
    r"F:\ERP_data\corpusRS05_50sample.xlsx",
    index=False
)

NameError: name 'idx' is not defined

In [10]:
# empty/deleted
import pandas as pd
threshold = 0.5

selected_idx = [
    i for i, p in enumerate(submission_prob)
    if p >= threshold
]

print("Threshold:", threshold)
print("Selected submissions:", len(selected_idx))

stats = {
    "normal_selftext": 0,
    "empty_selftext": 0,
    "deleted_selftext": 0,
    "removed_selftext": 0
}

empty_examples = []
for i in selected_idx:
    obj = objects[i]
    selftext = obj.get("selftext") or ""
    selftext_clean = selftext.strip()
    if selftext_clean == "":
        stats["empty_selftext"] += 1
        if len(empty_examples) < 10:
            empty_examples.append({
                "probability": round(submission_prob[i],3),
                "title": obj.get("title","")
            })
    elif selftext_clean == "[deleted]":
        stats["deleted_selftext"] += 1
    elif selftext_clean == "[removed]":
        stats["removed_selftext"] += 1
    else:
        stats["normal_selftext"] += 1

print("\nSelftext statistics:")
for k,v in stats.items():
    print(k, ":", v)
print("\nEmpty selftext examples:")
for x in empty_examples:
    print("\nProbability:", x["probability"])
    print("Title:", x["title"])

Threshold: 0.5
Selected submissions: 487

Selftext statistics:
normal_selftext : 214
empty_selftext : 190
deleted_selftext : 44
removed_selftext : 39

Empty selftext examples:

Probability: 0.553
Title: "Women studying gender studies instead of computer science is why Theresa May is Prime Minister"

Probability: 0.507
Title: Bronze Age Pervert on Twitter: "Either way, if you truly have dissident views, your career in whatever field, often even STEM would essentially be over. The only way to counter this would be with AGGRESSIVE anti-discrimination legislation such as exists now for favored minorities. "

Probability: 0.584
Title: Investigating The Stem Gender Equality Paradox - In Fairer Societies Fewer Woman Enter Science.

Probability: 0.511
Title: Hi. I’m an EdD student and beginning to formulate my dissertation topic/ questions. I want to do a qualitative study on coding/programming and how it affects middle school girls perception of STEM ability. I need 3 sub questions as well. A

In [11]:
def title_only_filter(title):
    title = title.lower().strip()
    if len(title.split()) < 5:
        return False
    if not ((M.search(title) or S.search(title))
            and G.search(title)):
        return False

    return True

In [14]:
import re
# M: Mathematics
M = re.compile(
    r"\b("
    r"maths|"
    r"mathematics|"
    r"algebra|"
    r"calculus|"
    r"geometry|"
    r"math\s+(class|major|degree|department|education)"
    r")\b",
    re.I
)

# S: STEM
S = re.compile(
    r"\b("
    r"stem|"
    r"engineering|"
    r"physics|"
    r"engineer|"
    r"computer\s*science|"
    r"cs\s*major|"
    r"programming|"
    r"coding|"
    r"coder|"
    r"software\s*engineer|"
    r"software\s*developer|"
    r"data\s*science|"
    r"data\s*scientist|"
    r"machine\s*learning|"
    r"artificial\s*intelligence|"
    r"women\s*in\s*tech|"
    r"girls\s*who\s*code"
    r")\b",
    re.I
)

# G: Gender
G = re.compile(
    r"\b("
    r"gender|"
    r"sexism|"
    r"sexist|"
    r"discrimination|"
    r"women|"
    r"woman|"
    r"girls|"
    r"female|"
    r"underrepresented|"
    r"male\s*dominated|"
    r"gender\s*gap"
    r")\b",
    re.I
)
keep_title_only = 0
remove_title_only = 0


for i in selected_idx:

    obj = objects[i]

    selftext = (obj.get("selftext") or "").strip()

    if selftext in ["", "[deleted]", "[removed]"]:

        title = obj.get("title","")

        if title_only_filter(title):
            keep_title_only += 1
        else:
            remove_title_only += 1


print("Keep title-only:", keep_title_only)
print("Remove title-only:", remove_title_only)

Keep title-only: 260
Remove title-only: 13


In [15]:
import random
title_only_idx = []
for i in selected_idx:
    obj = objects[i]
    selftext = (obj.get("selftext") or "").strip()
    if selftext in ["", "[deleted]", "[removed]"]:
        title = obj.get("title","")
        if title_only_filter(title):
            title_only_idx.append(i)
print("Title-only kept:", len(title_only_idx))

sample_idx = random.sample(
    title_only_idx,
    min(20, len(title_only_idx))
)


for n, i in enumerate(sample_idx, start=1):

    obj = objects[i]

    print("\n" + "="*80)
    print("Sample:", n)
    print("Probability:", round(submission_prob[i], 3))
    print("Subreddit:", obj.get("subreddit",""))
    print("Title:")
    print(obj.get("title",""))

Title-only kept: 260

Sample: 1
Probability: 0.676
Subreddit: MensRights
Title:
Isn't it funny that most of the women saying STEM fields are oppressive also refuse to actually try STEM first.

Sample: 2
Probability: 0.509
Subreddit: EngineeringStudents
Title:
I want to study engineering but also want to study women's studies?

Sample: 3
Probability: 0.561
Subreddit: toronto
Title:
Anybody want to go to Collision 2019 (Women In Tech) conference?

Sample: 4
Probability: 0.525
Subreddit: MarsSociety
Title:
All-woman engineering team heads to NASA Mars competition

Sample: 5
Probability: 0.572
Subreddit: u_Seagram1
Title:
Study finds that when all conditions are identical, female applicants are favored over male applicants by 2:1 for STEM tenure. This wouldn't be possible in a system with systemic male privilege!

Sample: 6
Probability: 0.754
Subreddit: xxstem
Title:
How to encourage women in STEM (x-post from /r/AskFeminists)

Sample: 7
Probability: 0.619
Subreddit: FindBacklink
Title:
Li

In [16]:
# 214normal selftext+260Title-only kept=474submission
import json
output_path = r"F:\ERP_data\final_corpusRS_05.jsonl"

threshold = 0.5
count = 0
normal_count = 0
title_only_count = 0

with open(output_path, "w", encoding="utf-8") as fout:
    for obj, p in zip(objects, submission_prob):
        # threshold filtering
        if p < threshold:
            continue
        title = obj.get("title") or ""
        selftext = (obj.get("selftext") or "").strip()
     
        # Case 1: normal selftext
        if selftext not in ["", "[deleted]", "[removed]"]:
            text_for_analysis = (
                title + " " + selftext
            )
            normal_count += 1
        
        # Case 2: title-only submission
        else:

            if title_only_filter(title):

                text_for_analysis = title

                title_only_count += 1

            else:
                continue


        # add fields for analysis
        obj["text_for_analysis"] = text_for_analysis
        obj["source_type"] = "submission"
        obj["classifier_probability"] = float(p)


        fout.write(
            json.dumps(
                obj,
                ensure_ascii=False
            ) + "\n"
        )

        count += 1



print("Saved:", output_path)
print("Total submissions:", count)
print("Normal selftext:", normal_count)
print("Title-only:", title_only_count)

Saved: F:\ERP_data\final_corpusRS_05.jsonl
Total submissions: 474
Normal selftext: 214
Title-only: 260
